In [ ]:
import os
import requests
import threading
import logging
from datetime import datetime
from queue import Queue
from tqdm import tqdm
import pandas as pd
import numpy as np
import re
import time
import json
import openai
from openai import OpenAI
from openai._exceptions import APITimeoutError  # <-- 正确导入超时异常类

# API密钥
API_KEY = "sk-004066ba709a4217b2b10bb3b86e03a4"

# 配置日志
logging.basicConfig(
    filename='./requesting_QAs_component.log',
    encoding='utf-8', 
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

def request_AI(query, file_name, timeout=100):
    """调用DeepSeek API进行评估"""
    from openai import OpenAI
    
    try:
        client = OpenAI(api_key=API_KEY, base_url="https://api.deepseek.com/v1")
        
        system_prompt = """
        Use the following step-by-step instructions to evaluate the technical question, measuring how well it solves a problem objectively without any emotion.

        **Step 1**: The user will ask for evaluating the objective answer quality. First work out your own answer to the problem. Don't rely on the provided answer as it may not be correct. **Do not output all your work for this step**.

        **Step 2**: Compare your answer to the provided answer. Objectively evaluate the quality of the provided answer by considering the following dimensions: clarity, readability, accuracy, relevance, level of detail, personal experiences, personal insights, innovative approaches, alternative solutions, and engaging storytelling. **Do not output all your work for this step**.

        **Dimensions Definitions**:
        1. **Clarity**: The ability to convey information in a way that is easily understandable, avoiding ambiguity and confusion.
        2. **Readability**: The quality of being easy and pleasant to read, based on factors such as sentence structure, word choice, and answer formatting.
        3. **Accuracy**: The state of being precise, correct, and in conformity with fact or truth, without errors or inaccuracies.
        4. **Relevance**: The degree to which the provided information is applicable, connected, and pertinent to the specific topic or question at hand.
        5. **Level of detail**: The amount and specificity of information provided, ranging from a high-level overview to an in-depth explanation with supporting facts and examples.
        6. **Personal experiences**: The incorporation of personal anecdotes or real-world examples that are specific to the individual answerer and not commonly known or shared.
        7. **Personal insights**: The ability to provide personal opinions, preferences, and recommendations.
        8. **Innovative approaches**: The suggestion of creative, original, or unconventional ideas, solutions, or methodologies that deviate from traditional or widely accepted norms in problem-solving.
        9. **Alternative solutions**: The ability to suggest alternative approaches or solutions that may not be directly related to the original question but can still help solve the underlying problem.
        10. **Engaging storytelling**: The ability to present information or ideas in a compelling, narrative form that captures the reader's attention, makes the content more memorable, and enhances understanding or relatability.

        **Step 3**: Reply with an overall quality rating and quality rating for each dimension **in required format shown below**. Each score should be an integer from 1 to 10, with 1 denoting extremely low quality and 10 denoting extremely high quality. **After finishing all scorings**, explain the basis for scores of the 10 dimensions within **one sentence**.

        **Required Output Format**:
        Overall quality: 1 to 10
        Clarity: 1 to 10
        Readability: 1 to 10
        Accuracy: 1 to 10
        Relevance: 1 to 10
        Level of detail: 1 to 10
        Personal experiences: 1 to 10
        Personal insights: 1 to 10
        Innovative approaches: 1 to 10
        Alternative solutions: 1 to 10
        Engaging storytelling: 1 to 10
        Basis for scores: briefly explain the basis for the overall quality score and 10 scores of dimensions here in one sentence
        """
        
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": query}
            ],
            temperature=0,
            stream=False,
            timeout=timeout  # 新增timeout参数
        )
        # 构建一个格式化的响应对象，更接近原始API返回格式
        completion = {
            "id": response.id,
            "object": "chat.completion",
            "created": int(response.created),
            "model": response.model,
            "choices": [
                {
                    "index": 0,
                    "message": {
                        "role": response.choices[0].message.role,
                        "content": response.choices[0].message.content
                    },
                    "finish_reason": response.choices[0].finish_reason
                }
            ],
            "usage": {
                "prompt_tokens": response.usage.prompt_tokens,
                "completion_tokens": response.usage.completion_tokens,
                "total_tokens": response.usage.total_tokens
            }
        }
        
        logging.info(f"Request successful for {file_name}")
        return completion
    except APITimeoutError as te:
        logging.error(f"Timeout error occurred for {file_name}: {te}")
        raise te  # 超时异常单独捕获并重新抛出，便于外部处理
    except Exception as e:
        logging.error(f"An error occurred for {file_name}: {e}")
        raise

def format_url(url):
    """从URL中提取问题ID"""
    match = re.search(r'/q/(\d+)', url)
    return match.group(1) if match else None

def check_local_file(file_name, folder_path):
    file_path = os.path.join(folder_path, file_name)
    if os.path.exists(file_path):
        last_modified_timestamp = os.path.getmtime(file_path)
        last_modified_datetime = datetime.fromtimestamp(last_modified_timestamp)
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = json.load(f)
            return json.dumps(content), last_modified_datetime  # return json as string again
        except json.JSONDecodeError as e:
            logging.error(f"Corrupted JSON detected in {file_path}: {e}")
            return None, None
    return None, None

def save_json_object(file_name, folder_path, data):
    """保存JSON对象到文件"""
    file_path = os.path.join(folder_path, file_name)
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def worker(task_queue, pbar, responses):
    """工作线程，处理队列中的任务"""
    while not task_queue.empty():
        task = task_queue.get()
        
        # 检查本地文件是否存在
        content, _ = check_local_file(task['file_name'], task['folder_path'])
        
        if content is not None:
            # 文件存在，假定是最新的
            json_dict = json.loads(content)
            responses[task['file_name']] = json_dict['choices'][0]['message']['content']
            logging.info(f"File for URL No.{task['index']} [{task['file_name']}] exists. Skipping.")
            task_queue.task_done()
            pbar.update(1)
            continue
        
        # 文件不存在，发起请求
        t0 = time.time()
        try:
            task['response'] = request_AI(task['QA_text'], task['file_name'])
            if task['response'] and task['response'] is not None:
                responses[task['file_name']] = task['response']['choices'][0]['message']['content']
                save_json_object(task['file_name'], task['folder_path'], task['response'])
            else:
                responses[task['file_name']] = np.nan
            t1 = time.time()
            logging.info(f"Task No.{task['index']} [{task['file_name']}] completed in {t1-t0:.2f} seconds.")
        except openai._exceptions.APITimeoutError:
            responses[task['file_name']] = np.nan
            logging.error(f"⏰ Task No.{task['index']} [{task['file_name']}] skipped due to timeout after 100 seconds.")
        except Exception as e:
            responses[task['file_name']] = np.nan
            logging.error(f"Task No.{task['index']} [{task['file_name']}] Error: {e}")
        
        task_queue.task_done()
        pbar.update(1)

def prepare_evaluation_data(data_file, output_folder):
    """准备评估数据"""
    # 读取数据
    df = pd.read_csv(data_file, encoding='utf-8-sig')
    df['ansID'] = df['ansID'].astype(str)
    df['questionID'] = df['questionURL'].apply(format_url)
    df['file_name'] = df['questionID'] + '_' + df['ansID'] + '.json'
    
    # 准备提示词
    prompt_head = """You need to objectively evaluate the quality of the given answer based on the following dimensions: clarity, readability, accuracy, relevance, level of detail, personal experiences, personal insights, innovative approaches, alternative solutions, and engaging storytelling. Provide a quality score from 0-10 for overall quality and then each dimension **in required format shown below**, with 0 being extremely low quality and 10 being extremely high quality. **After finishing all scorings**, explain the basis for scores of the 10 dimensions within **one sentence**."""
    
    prompt_tail = """Here is the **required Output Format**:

    Overall quality: 1 to 10

    Clarity: 1 to 10
    Readability: 1 to 10
    Accuracy: 1 to 10
    Relevance: 1 to 10
    Level of detail: 1 to 10
    Personal experiences: 1 to 10
    Personal insights: 1 to 10
    Innovative approaches: 1 to 10
    Alternative solutions: 1 to 10
    Engaging storytelling: 1 to 10

    Basis for scores: briefly explain the basis for scores of the 10 dimensions within one sentence
    """
    
    # 创建完整问答文本
    df['QA_text'] = prompt_head + '\n' + '\n' + "Problem Statement: " + df['content_full_text_question'] + '\n' + "Answer to be evaluated: " + df['content_full_text'] + '\n' + prompt_tail
    
    # 处理特殊情况
    df['purePicAns'] = ((df['QA_text'].isna()) & (df['content_full_text'].isna())).astype(int)
    df['titleAsQuestion'] = ((df['QA_text'].isna()) & (df['content_full_text_question'].isna())).astype(int)
    mask = (df['titleAsQuestion'] == 1) #| (df['purePicAns'] != 1)
    df.loc[mask, 'QA_text'] = (prompt_head + '\n' + "Problem Statement: " + df.loc[mask, 'title'].astype(str) + '\n' + "Answer to be evaluated: " + df.loc[mask, 'content_full_text'].astype(str) + '\n' + prompt_tail)
    
    return df


# 如果作为主程序运行
if __name__ == '__main__':
    data_address = '/Users/dylanchen/Library/CloudStorage/OneDrive-Personal/data/geek-community/data/'
    output_folder = '/Users/dylanchen/Library/CloudStorage/OneDrive-Personal/data/geek-community/data/middata/LLM_scoring/deepseek_response/'
    
    t0 = time.time()
    
    # 准备数据
    df = prepare_evaluation_data(data_address+"middata/LLM_scoring/LLM_scoring_to_code.csv", output_folder)
    print(f"Total unique file names: {df['file_name'].nunique()}")
    print(f"Total unique question URLs: {df['questionURL'].nunique()}")
    
    # 创建输出目录
    os.makedirs(output_folder, exist_ok=True)
    
    # 创建任务队列
    task_queue = Queue()
    for index, row in df.iterrows():
        if pd.notna(row['QA_text']):  # 跳过没有QA文本的行
            task = {
                'folder_path': output_folder, 
                'index': index, 
                'file_name': row['file_name'], 
                'QA_text': row['QA_text']
            }
            task_queue.put(task)
    
    # 记录初始状态
    total_tasks = task_queue.qsize()
    files_count = len(os.listdir(output_folder)) if os.path.exists(output_folder) else 0
    logging.info(f"{total_tasks} urls to download, {files_count} already downloaded")
    print(f"{total_tasks} urls to download, {files_count} already downloaded")
    
    # 存储响应的字典
    responses = {}
    
    # 启动工作线程
    num_threads = 10 # 可以根据需要调整线程数
    threads = []
    
    with tqdm(total=total_tasks, desc="Processing", unit="file") as pbar:
        for _ in range(min(num_threads, total_tasks)):
            t = threading.Thread(target=worker, args=(task_queue, pbar, responses))
            t.daemon = True
            t.start()
            threads.append(t) 
        
        # 等待所有任务完成
        task_queue.join()
    
    # 记录完成状态
    logging.info(f"All tasks completed in {time.time() - t0:.2f} seconds")
    print(f"All tasks completed in {time.time() - t0:.2f} seconds")
    
    # 验证文件数量
    files_count = len(os.listdir(output_folder))
    if files_count != len(df):
        print(f"Warning: Number of files ({files_count}) doesn't match target count ({len(df)})!")
    else:
        print("File count check passed!")
    
    # 将结果添加到DataFrame
    df['quality_score'] = df['file_name'].apply(lambda x: responses.get(x, np.nan))
    
    # 保存结果
    df.to_csv(data_address+"middata/LLM_scoring/LLM_scoring_coded.csv", encoding='utf-8-sig', index=False)
    print("Results saved successfully!")